# ST-OMR Meter V5-2 — 1200 TRAIN Full-Meter BBox

**Canonical continuation:** V5-1’de kabul edilen 30 kırmızı full-meter BBox değişmeden korunur. Kalan 1170 TRAIN örneğinde aynı kural devam eder: tek kutu üst+alt meter rakamlarını birlikte kapsar. **BBox ikiye bölünmez; numerator/denominator kutusu üretilmez.** VAL ve FINAL HOLDOUT kapalıdır.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, pathlib, subprocess, sys, shutil
EXPECTED_CODE_SHA = 'de46b6c163376a0bab6c6ac768bca6af76c7afa5'
BRANCH = 'agent/meter-v5-2-train-bbox-scale-contract'
REPO_URL = 'https://github.com/khfy7wpr5p-maker/st-omr-training.git'
REPO_DIR = pathlib.Path('/content/st-omr-training-v5-2')
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git','clone','--depth','30','--branch',BRANCH,REPO_URL,str(REPO_DIR)], check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach',EXPECTED_CODE_SHA], check=True)
actual = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'], text=True).strip()
assert actual == EXPECTED_CODE_SHA, (actual, EXPECTED_CODE_SHA)
subprocess.run([sys.executable,'-m','pip','install','-q','Pillow==12.3.0'], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
print('CODE_PIN=PASS', actual)


## Güvenli precheck — mevcut 30 seed + 1200 TRAIN bağlama

Bu hücre exact clean dataset’i, V5-1 seed hash’lerini ve 1200 TRAIN image binding’ini doğrular. İlk 30 kabul edilmiş BBox değiştirilmeden yeni 1200 kayıt dosyasına seed edilir. Uzun sürebileceği için 10 saniyelik heartbeat gösterilir. Model/training/inference çalışmaz.


In [ ]:
import pathlib, threading, time
from st_omr_training.meter_v5_2_train_bbox_scale import ScaleAnnotationSession, TRAIN_TOTAL
MYDRIVE = pathlib.Path('/content/drive/MyDrive')
DATA_ROOT = MYDRIVE / 'TEST' / 'METER_V2_1500_PACKAGE_AB_CLEAN'
if not MYDRIVE.is_dir(): raise RuntimeError(f'MyDrive not mounted: {MYDRIVE}')
if not DATA_ROOT.is_dir(): raise RuntimeError(f'Authoritative dataset not found: {DATA_ROOT}')
print('===== METER V5-2 LIVE PRECHECK =====')
print('authority=', DATA_ROOT)
_state = {'phase':'construct_scale_session','session':None,'error':None}
def _worker():
    try:
        _state['session'] = ScaleAnnotationSession(data_root=DATA_ROOT)
        _state['phase'] = 'done'
    except BaseException as exc:
        _state['error'] = exc
        _state['phase'] = 'error'
_thread = threading.Thread(target=_worker, name='meter-v5-2-precheck', daemon=False)
_started = time.monotonic()
_thread.start()
while _thread.is_alive():
    elapsed = int(time.monotonic() - _started)
    print(f'[LIVE] phase={_state["phase"]} elapsed={elapsed}s | FINAL_HOLDOUT=LOCKED | TRAINING=CLOSED | MODEL=CLOSED | INFERENCE=0', flush=True)
    _thread.join(timeout=10)
_thread.join()
if _state['error'] is not None:
    print('[PRECHECK] FAIL-CLOSED:', repr(_state['error']))
    raise _state['error']
SESSION = _state['session']
assert len(SESSION.samples) == TRAIN_TOTAL == 1200
assert SESSION.handled_count >= 30
assert SESSION.pass_count >= 30
assert SESSION.resume_index() >= 30
print('PRECHECK=PASS')
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('resume_index=', SESSION.resume_index())
print('FIRST_30_SEEDS=LOCKED; BBOX_SPLIT=False; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## 1200 TRAIN BBox arayüzü

Arayüz kaldığın ilk işlenmemiş TRAIN örneğinden açılır. İlk 30 seed kilitlidir. Her yeni örnekte yalnız **tek full-meter kırmızı kutu** çizilir.


In [ ]:
from st_omr_training.meter_v5_2_train_bbox_colab import launch_colab_scale
SESSION = launch_colab_scale(data_root=str(DATA_ROOT), session=SESSION)
print('V5_2_UI=READY')
print('resume_index=', SESSION.resume_index())
print('handled=', SESSION.handled_count, 'pass=', SESSION.pass_count, 'review=', SESSION.review_count)
print('FINAL_HOLDOUT=LOCKED; TRAINING=False; MODEL_OPENED=False; INFERENCE_COUNT=0')


## Mekanik audit — 1200/1200 tamamlandıktan sonra

Audit insan BBox’larını değiştirmez. 1200 PASS, 400/class, seed mutation=0 ve diğer mekanik kapıları doğrular. PASS olsa bile training otomatik açılmaz; ayrıca insan görsel QA gerekir.


In [ ]:
import json
from st_omr_training.meter_v5_2_train_bbox_scale import write_train_audit
AUDIT_PATH = write_train_audit(DATA_ROOT)
AUDIT = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
print(json.dumps(AUDIT, indent=2, sort_keys=True))
print('AUDIT_PATH=', AUDIT_PATH)
print('TRAINING_AUTHORIZED=', AUDIT['training_authorized'])
print('HUMAN_VISUAL_REVIEW_REQUIRED=', AUDIT['human_visual_review_required'])
